In [0]:
%sql
DELETE FROM food_inspection.silver.chicago_inspections
WHERE inspection_id LIKE 'TEST_SCD2%';

In [0]:
%sql
DELETE FROM food_inspection.gold.dim_establishment
WHERE establishment_name LIKE 'SCD_TEST_%';

In [0]:
catalog_name = "food_inspection"
silver_schema = "silver"
gold_schema = "gold"

print("Setup complete")

In [0]:
silver_df = spark.table(f"{catalog_name}.{silver_schema}.chicago_inspections")

print("Schema loaded")

In [0]:
from datetime import date

initial_row = spark.createDataFrame(
    [(
        "TEST_SCD2_1",              # inspection_id
        "SCD_TEST_STORE",          # dba_name
        "SCD_TEST_STORE",          # aka_name
        None,                      # license_number
        "Restaurant",              # facility_type (INITIAL)
        "Risk 3 (Low)",            # risk
        "123 Test St",             # address
        None,                      # city
        None,                      # state
        "60601",                   # zip_code
        date(2025,1,1),            # inspection_date
        "TEST",                    # inspection_type
        "Pass",                    # results
        85,                        # violation_score
        None,                      # latitude
        None,                      # longitude
        False                      # is_out_of_area
    )],
    schema=silver_df.schema
)

initial_row.write.mode("append").saveAsTable(f"{catalog_name}.{silver_schema}.chicago_inspections")

print("Initial record inserted")

In [0]:
%sql
SELECT inspection_id, dba_name, facility_type
FROM food_inspection.silver.chicago_inspections
WHERE inspection_id LIKE 'TEST_SCD2%'

In [0]:
## run gold layer code

In [0]:
%sql
SELECT establishment_name, facility_type, is_current
FROM food_inspection.gold.dim_establishment
WHERE establishment_name = 'SCD_TEST_STORE'

In [0]:
changed_row = spark.createDataFrame(
    [(
        "TEST_SCD2_2",              # inspection_id
        "SCD_TEST_STORE",
        "SCD_TEST_STORE",
        None,
        "Grocery Store",            # CHANGE
        "Risk 3 (Low)",
        "123 Test St",
        None,
        None,
        "60601",
        date(2025,2,1),             # later date
        "TEST",
        "Pass",
        90,
        None,
        None,
        False
    )],
    schema=silver_df.schema
)

changed_row.write.mode("append").saveAsTable(f"{catalog_name}.{silver_schema}.chicago_inspections")

print("Changed record inserted")

In [0]:
%sql
SELECT inspection_id, dba_name, facility_type
FROM food_inspection.silver.chicago_inspections
WHERE inspection_id LIKE 'TEST_SCD2%'
ORDER BY inspection_date

In [0]:
## run gold layer again

In [0]:
%sql
SELECT 
    establishment_name,
    facility_type,
    is_current,
    effective_start_date,
    effective_end_date
FROM food_inspection.gold.dim_establishment
WHERE establishment_name = 'SCD_TEST_STORE'
ORDER BY effective_start_date

In [0]:
%sql
SELECT 
    establishment_name,
    facility_type,
    effective_start_date,
    effective_end_date,
    CASE 
        WHEN is_current THEN 'ACTIVE'
        ELSE 'HISTORICAL'
    END AS status
FROM food_inspection.gold.dim_establishment
WHERE establishment_name = 'SCD_TEST_STORE'
ORDER BY effective_start_date